# Phase 1 - Group-aware, leakage-free split (Kaggle notebook)

Splits the **ArSL `Mosl_alphabet`** dataset (32 classes) **once** and saves the
per-split *image path lists* to `/kaggle/working`. The split is **group-aware**:
near-duplicate frames are clustered (perceptual hash) and every cluster is kept
entirely inside one split, so no duplicate leaks across train/val/test
(CLAUDE.md constraint #1).

**Output** (run once): `train.csv`, `val.csv`, `test.csv` with columns
`image_path,label,cluster_id`, where `image_path` is **relative** to the dataset
root. At training time you just read these CSVs and open
`DATA_ROOT / image_path` - you never re-split.

This notebook is self-contained (mirrors `src/data/`); it needs only
`imagehash`. Run the cells top to bottom.

In [ ]:
!pip -q install imagehash

## 1. Config
Edit these if paths/params change. `DATA_ROOT` is the read-only Kaggle mount.

In [ ]:
from pathlib import Path

# ---- EDIT HERE -----------------------------------------------------------
DATA_ROOT  = Path("/kaggle/input/datasets/youssefnouiouar1/SLR/Mosl_alphabet")
OUTPUT_DIR = Path("/kaggle/working/xaislr/splits")

CLASSES = ["ain", "al", "aleff", "bb", "dal", "dha", "dhad", "fa", "gaaf",
           "ghain", "ha", "haa", "jeem", "kaaf", "khaa", "la", "laam", "meem",
           "nun", "ra", "saad", "seen", "sheen", "ta", "taa", "thaa", "thal",
           "toot", "waw", "ya", "yaa", "zay"]

RATIOS   = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED     = 0
STRATIFY = True          # per-class 70/15/15 while keeping clusters intact

# dedup (near-duplicate control)
HASH_SIZE      = 16      # phash grid -> 16*16 = 256-bit hash
HAMMING_THRESH = 6       # <= this many differing bits => same duplicate cluster
NUM_BANDS      = None    # None => HAMMING_THRESH + 1 (LSH pigeonhole guarantee)
NUM_WORKERS    = 4       # hashing processes; set 1 if you hit any issue
# --------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Help locate the dataset if the path is off.
if not DATA_ROOT.exists():
    print("DATA_ROOT not found:", DATA_ROOT)
    base = Path("/kaggle/input")
    if base.exists():
        print("Available under /kaggle/input:")
        for p in sorted(base.rglob("*")):
            if p.is_dir():
                print("  ", p)
    raise FileNotFoundError("Fix DATA_ROOT above to point at the Mosl_alphabet folder.")
print("DATA_ROOT OK ->", DATA_ROOT)

## 2. Scan the dataset
One row per image; `image_path` is stored **relative** to `DATA_ROOT`.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
import imagehash
from concurrent.futures import ProcessPoolExecutor
from itertools import repeat
from collections import defaultdict
from tqdm.auto import tqdm

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".tif", ".tiff", ".webp"}

def scan_dataset(root, classes):
    root = Path(root)
    subdirs = sorted(d.name for d in root.iterdir() if d.is_dir())
    missing = [c for c in classes if c not in subdirs]
    extra   = [d for d in subdirs if d not in classes]
    rows = []
    for cls in classes:
        cdir = root / cls
        if not cdir.is_dir():
            continue
        for f in sorted(cdir.iterdir()):
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                rows.append((f"{cls}/{f.name}", cls))   # POSIX-relative path
    df = pd.DataFrame(rows, columns=["image_path", "label"])
    return df, {"subdirs": subdirs, "missing": missing, "extra": extra}

df, info = scan_dataset(DATA_ROOT, CLASSES)
print("images found:", len(df))
if info["missing"]: print("WARNING missing class folders:", info["missing"])
if info["extra"]:   print("WARNING extra folders (ignored):", info["extra"])
assert len(df) > 0, "No images found - check DATA_ROOT / CLASSES."
df.head()

## 3. Near-duplicate clustering (perceptual hash)
Assigns a `cluster_id` to every image. Exact duplicates are collapsed first,
then LSH banding (pigeonhole: `HAMMING_THRESH + 1` bands) finds near-duplicate
candidates without an O(N^2) all-pairs scan.

In [ ]:
class DSU:
    """Union-find with path halving + union by rank."""
    def __init__(self, n): self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x: self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1

def _hash_one(abs_path, hash_size):
    try:
        with Image.open(abs_path) as im:
            h = imagehash.phash(im, hash_size=hash_size)
    except Exception:
        return None                      # unreadable image -> dropped later
    return np.packbits(h.hash.flatten()).tobytes()

def compute_hashes(abs_paths, hash_size, num_workers):
    packed = [None] * len(abs_paths)
    if num_workers and num_workers > 1:
        with ProcessPoolExecutor(max_workers=num_workers) as ex:
            it = ex.map(_hash_one, abs_paths, repeat(hash_size), chunksize=64)
            for i, res in enumerate(tqdm(it, total=len(abs_paths), desc="phash")):
                packed[i] = res
    else:
        for i, p in enumerate(tqdm(abs_paths, desc="phash")):
            packed[i] = _hash_one(p, hash_size)
    return packed

def cluster_duplicates(packed, hash_size, threshold, num_bands=None):
    L = hash_size * hash_size
    n = len(packed)
    dsu = DSU(n)
    bits = [np.unpackbits(np.frombuffer(p, dtype=np.uint8))[:L] for p in packed]

    # 1) collapse exact duplicates, keep one representative per distinct hash
    rep_of, reps = {}, []
    for i in range(n):
        r = rep_of.get(packed[i])
        if r is None: rep_of[packed[i]] = i; reps.append(i)
        else:         dsu.union(r, i)

    # 2) LSH banding over distinct reps
    b = num_bands or (threshold + 1)
    b = max(1, min(b, L))
    for band in np.array_split(np.arange(L), b):
        buckets = {}
        for r in reps:
            buckets.setdefault(bits[r][band].tobytes(), []).append(r)
        for mem in buckets.values():
            m = len(mem)
            if m < 2: continue
            for a in range(m):
                ba = bits[mem[a]]
                for c in range(a + 1, m):
                    if np.count_nonzero(ba != bits[mem[c]]) <= threshold:
                        dsu.union(mem[a], mem[c])

    # 3) compact cluster ids in first-seen order (deterministic)
    root_to_id, cid, nxt = {}, np.empty(n, dtype=np.int64), 0
    for i in range(n):
        root = dsu.find(i)
        if root not in root_to_id: root_to_id[root] = nxt; nxt += 1
        cid[i] = root_to_id[root]
    return cid

abs_paths = [str(DATA_ROOT / p) for p in df["image_path"]]
packed = compute_hashes(abs_paths, HASH_SIZE, NUM_WORKERS)

valid = np.array([p is not None for p in packed])
n_dropped = int((~valid).sum())
df = df.loc[valid].reset_index(drop=True).copy()
df["cluster_id"] = cluster_duplicates(
    [p for p, ok in zip(packed, valid) if ok], HASH_SIZE, HAMMING_THRESH, NUM_BANDS)

print(f"kept {len(df)} images | dropped {n_dropped} unreadable | "
      f"{df['cluster_id'].nunique()} clusters")

## 4. Group-aware split + leakage check
Whole clusters go to one split; per class we greedily fill whichever split is
furthest below its image target (approx stratified 70/15/15).

In [ ]:
def group_aware_split(df, ratios, seed=0, stratify=True):
    assert abs(sum(ratios.values()) - 1.0) < 1e-6, "ratios must sum to 1"
    rng = np.random.default_rng(seed)
    g = df.groupby("cluster_id")
    sizes = g.size().to_dict()
    dom   = g["label"].agg(lambda s: s.mode().iloc[0]).to_dict()  # dominant label
    assign = {}

    def alloc(cids):
        total = sum(sizes[c] for c in cids)
        tgt = {s: r * total for s, r in ratios.items()}
        got = {s: 0.0 for s in ratios}
        tie = {c: rng.random() for c in cids}
        for c in sorted(cids, key=lambda c: (-sizes[c], tie[c])):   # largest first
            s = max(ratios, key=lambda s: tgt[s] - got[s])          # biggest deficit
            assign[c] = s
            got[s] += sizes[c]

    if stratify:
        by = defaultdict(list)
        for c, l in dom.items(): by[l].append(c)
        for cids in by.values(): alloc(cids)
    else:
        alloc(list(sizes))

    out = df.copy()
    out["split"] = out["cluster_id"].map(assign)
    return out

df = group_aware_split(df, RATIOS, SEED, STRATIFY)

# leakage check: no cluster may span two splits
spans = df.groupby("cluster_id")["split"].nunique()
n_leak = int((spans > 1).sum())
assert n_leak == 0, f"LEAKAGE: {n_leak} clusters span multiple splits"
print("LEAKAGE CHECK: PASS - no cluster spans two splits")

## 5. Save the per-split path lists (run once)
These CSVs are the deliverable - reuse them at training time.

In [ ]:
import json, datetime

cols = ["image_path", "label", "cluster_id"]
for s in ["train", "val", "test"]:
    out_csv = OUTPUT_DIR / f"{s}.csv"
    df.loc[df["split"] == s, cols].to_csv(out_csv, index=False)
    print(f"wrote {out_csv}  ({int((df['split'] == s).sum())} rows)")

meta = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "seed": SEED, "data_root": str(DATA_ROOT), "num_classes": len(CLASSES),
    "images_kept": int(len(df)), "images_dropped": n_dropped,
    "num_clusters": int(df["cluster_id"].nunique()), "ratios": RATIOS,
    "hash_size": HASH_SIZE, "hamming_threshold": HAMMING_THRESH,
}
(OUTPUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
print("wrote", OUTPUT_DIR / "meta.json")

## 6. Sanity report

In [ ]:
sizes = df.groupby("cluster_id").size()
print("images kept:      ", len(df))
print("total clusters:   ", sizes.shape[0])
print("duplicate clusters:", int((sizes > 1).sum()),
      f"({int(sizes[sizes > 1].sum())} images, "
      f"{100 * int(sizes[sizes > 1].sum()) / len(df):.1f}% of kept)")
print("largest cluster:  ", int(sizes.max()), "images\n")

tot = df["split"].value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)
for s in ["train", "val", "test"]:
    n = int(tot[s]); print(f"{s:5s}: {n:7d}  ({100 * n / len(df):.1f}%)")

pivot = (df.groupby(["label", "split"]).size().unstack(fill_value=0)
           .reindex(columns=["train", "val", "test"], fill_value=0))
pivot["total"] = pivot.sum(axis=1)
pivot

## 7. How to use the splits at training time
Read the CSV for the split you want and open each image from `DATA_ROOT`. The
cell below proves the saved paths resolve to real images.

In [ ]:
# quick check: the saved relative paths open correctly from DATA_ROOT
for s in ["train", "val", "test"]:
    part = pd.read_csv(OUTPUT_DIR / f"{s}.csv")
    rel = part["image_path"].iloc[0]
    im = Image.open(DATA_ROOT / rel).convert("RGB")
    print(f"{s:5s} n={len(part):6d}  first={rel}  size={im.size}")

# Example training-time usage:
#   train_df = pd.read_csv(OUTPUT_DIR / "train.csv")
#   for _, row in train_df.iterrows():
#       img = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
#       label = row["label"]